# New V4

In [30]:
pip install -q segmentation-models-pytorch timm

# Import Libraries

In [31]:
import os
import zipfile
import time
import glob
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
from PIL import Image
from community import community_louvain
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from dataclasses import dataclass

# Config Class

In [32]:
from dataclasses import dataclass
import torch

@dataclass
class Config:

    # =====================================================
    # Dataset
    # =====================================================
    DATASET_ZIP: str = "/content/kvasir-seg.zip"
    TRAIN_SPLIT: float = 0.80

    # =====================================================
    # Image Processing
    # =====================================================
    IMAGE_SIZE: tuple = (256, 256)

    # =====================================================
    # DataLoader
    # =====================================================
    BATCH_SIZE: int = 16
    NUM_WORKERS: int = 2
    PIN_MEMORY: bool = True

    # =====================================================
    # Model
    # =====================================================
    MODEL_NAME: str = "UNet++"

    ENCODER_NAME: str = "resnet34"
    ENCODER_WEIGHTS: str = "imagenet"

    IN_CHANNELS: int = 3
    NUM_CLASSES: int = 1

    # =====================================================
    # Training
    # =====================================================
    EPOCHS: int = 50              # Final: 100–200
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 1e-4

    OPTIMIZER: str = "AdamW"
    LOSS_FUNCTION: str = "BCE + Dice"

    RANDOM_SEED: int = 42

    # =====================================================
    # Graph Pruning
    # =====================================================
    SIMILARITY_THRESHOLD: float = 0.70
    RETENTION_RATIO: float = 0.20

    # =====================================================
    # DINOv2
    # =====================================================
    DINO_MODEL: str = "dinov2_vits14"

    # =====================================================
    # Evaluation
    # =====================================================
    THRESHOLD: float = 0.5

    # =====================================================
    # Device
    # =====================================================
    DEVICE: torch.device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    # =====================================================
    # Save
    # =====================================================
    SAVE_FEATURES: bool = True
    FEATURE_FILE: str = "dinov2_features.npy"

    SAVE_PRUNED_INDICES: bool = True
    PRUNED_INDEX_FILE: str = "pruned_indices.npy"

    SAVE_MODEL: bool = True
    MODEL_DIR: str = "checkpoints"

    BEST_MODEL_NAME: str = "best_unetpp.pth"

    # =====================================================
    # Debug
    # =====================================================
    DEBUG: bool = False
    DEBUG_SAMPLES: int = 200


config = Config()

# MODULE 1: DATASET PREPARATION & HANDLING

In [33]:
# =====================================================================
# MODULE 1: DATASET PREPARATION & HANDLING
# =====================================================================

class KvasirSegDataset(Dataset):
    """Custom Dataset wrapper for parsing unzipped Kvasir-SEG contents."""
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = sorted(image_paths)
        self.mask_paths = sorted(mask_paths)
        self.transform = transform

        # Internal transformation to force geometric uniformity for tensors
        self.base_tensor_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Grayscale for ground truth

        if self.transform:
            # Apply identical random states to both images and targets if augmenting
            seed = np.random.randint(2147483647)
            torch.manual_seed(seed)
            image_t = self.transform(image)
            torch.manual_seed(seed)
            mask_t = self.transform(mask)
        else:
            image_t = self.base_tensor_transform(image)
            mask_t = self.base_tensor_transform(mask)

        # Binarize the mask explicitly to 0.0 or 1.0
        mask_t = (mask_t > 0.5).float()
        return image_t, mask_t


def prepare_kvasir_data(zip_path, extract_to='./kvasir_data'):
    """Extracts zip directory structures securely into distinct clean paths."""
    if not os.path.exists(extract_to):
        print(f"[Data] Extracting dataset archive: {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)

    # Locate paths dynamically based on common Kaggle directory naming structures
    search_img = os.path.join(extract_to, "**", "images", "*.jpg")
    search_masks = os.path.join(extract_to, "**", "masks", "*.jpg")

    # Fallback to PNG extensions if present
    img_paths = glob.glob(search_img, recursive=True) or glob.glob(os.path.join(extract_to, "**", "images", "*.png"), recursive=True)
    mask_paths = glob.glob(search_masks, recursive=True) or glob.glob(os.path.join(extract_to, "**", "masks", "*.png"), recursive=True)

    print(f"[Data] Found {len(img_paths)} images and {len(mask_paths)} segmentation masks.")
    return img_paths, mask_paths

# MODULE 2: DINOv2 MANIFOLD EXTRACTION & PRUNING ENGINE

In [34]:
# =====================================================================
# MODULE 2: DINOv2 MANIFOLD EXTRACTION & PRUNING ENGINE
# =====================================================================

class DinoEmbeddingGraphPruner:
    """Extracts DINOv2 visual features and runs Louvain community selection."""
    def __init__(self, dataset: Dataset, train_indices: list, device: torch.device):
        self.dataset = dataset
        self.train_indices = train_indices
        self.device = device

        print("[DINOv2 Engine] Initializing Frozen ViT Backbone (dinov2_vits14)...")
        self.model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
        self.model.eval()

        # Standard ImageNet scaling parameters required by DINOv2
        self.dino_transform = transforms.Compose([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    @torch.no_grad()
    def prune(self, tau: float = 0.65, p: float = 0.20) -> list:
        embeddings = []
        print("[DINOv2 Engine] Computing Semantic Latent Spaces for Training Partition...")

        for idx in self.train_indices:
            img, _ = self.dataset[idx]
            # Input shape: [3, 224, 224] -> normalized and batched
            input_tensor = self.dino_transform(img).unsqueeze(0).to(self.device)

            # Extract Global CLS Token Vector
            cls_vec = self.model(input_tensor)
            cls_vec = nn.functional.normalize(cls_vec, p=2, dim=1)
            embeddings.append(cls_vec.cpu().numpy().flatten())

        embeddings = np.array(embeddings) # Matrix of shape [N_train, 384]

        # Generate the continuous manifold using Cosine Similarity Dot Products
        cosine_sim_matrix = np.dot(embeddings, embeddings.T)

        # Map values from [-1, 1] securely to an adjacency range of [0, 1]
        adj_matrix = (cosine_sim_matrix + 1.0) / 2.0

        # Structural sparsification via threshold mapping
        binary_adj = (adj_matrix >= tau).astype(int)

        # Build topological graph space
        G = nx.from_numpy_array(binary_adj)

        print("[DINOv2 Engine] Solving Network Modularity via Louvain Partitions...")
        partition = community_louvain.best_partition(G)

        communities = {}
        for node, comm_id in partition.items():
            communities.setdefault(comm_id, []).append(node)

        selected_local_nodes = []
        for comm_id, nodes in communities.items():
            if len(nodes) <= 1:
                selected_local_nodes.append(nodes[0])
            else:
                # Rank local cluster frames based on their node degree centrality
                degrees = dict(G.degree(nodes))
                sorted_nodes = sorted(nodes, key=lambda n: degrees[n], reverse=True)

                budget = max(1, int(np.ceil(p * len(nodes))))
                selected_local_nodes.extend(sorted_nodes[:budget])

        print(f"Number of communities: {len(communities)}")
        sizes = [len(nodes) for nodes in communities.values()]
        print(f"Min size : {min(sizes)}")
        print(f"Max size : {max(sizes)}")
        print(f"Average  : {np.mean(sizes):.2f}")

        # Remap localized positions back to global dataset index coordinates
        pruned_global_indices = [self.train_indices[i] for i in selected_local_nodes]
        print(f"[DINOv2 Engine] Pruning complete. Budget reduced from {len(self.train_indices)} to {len(pruned_global_indices)} samples.")
        return pruned_global_indices

# MODULE 3: SEGMENTATION ARCHITECTURE & PIPELINE TRAINING

In [35]:

# =====================================================================
# MODULE 3: SEGMENTATION ARCHITECTURE & PIPELINE TRAINING
# =====================================================================

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class CompactUNet(nn.Module):
    """Standard U-Net encoder-decoder network optimized for training validation."""
    def __init__(self, in_c=3, out_c=1, features=[64, 128, 256]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)

        for feat in features:
            self.downs.append(DoubleConv(in_c, feat))
            in_c = feat

        for feat in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feat*2, feat, 2, stride=2))
            self.ups.append(DoubleConv(feat*2, feat))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final = nn.Conv2d(features[0], out_c, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip = skips[idx//2]
            if x.shape != skip.shape:
                x = torch.nn.functional.interpolate(x, size=skip.shape[2:])
            x = self.ups[idx+1](torch.cat((skip, x), dim=1))
        return self.final(x)


class ExecutionEngine:
    """Handles standard PyTorch loops, metrics calculation, and evaluation."""
    def __init__(self, model, device):
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-3)

    def stats(self, pred, target):
        preds = (torch.sigmoid(pred) > 0.5).float()
        inter = (preds * target).sum(dim=(2, 3))
        total = preds.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = ((2.0 * inter) / (total + 1e-5)).mean().item()
        iou = ((inter + 1e-5) / (total - inter + 1e-5)).mean().item()
        return dice, iou

    def fit(self, loader):
        self.model.train()
        total_loss = 0.0
        for imgs, masks in loader:
            imgs, masks = imgs.to(self.device), masks.to(self.device)
            self.optimizer.zero_grad()
            loss = self.criterion(self.model(imgs), masks)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(loader)

    # def evaluate(self, loader):
    #     self.model.eval()
    #     d, i = 0.0, 0.0
    #     with torch.no_grad():
    #         for imgs, masks in loader:
    #             imgs, masks = imgs.to(self.device), masks.to(self.device)
    #             md, mi = self.stats(self.model(imgs), masks)
    #             d += md * imgs.size(0)
    #             i += mi * imgs.size(0)
    #     return d / len(loader.dataset), i / len(loader.dataset)

    @torch.no_grad()
    def evaluate(self, loader):

        self.model.eval()

        total_dice = 0
        total_iou = 0
        total_correct = 0
        total_pixels = 0

        for images, masks in loader:

            images = images.to(self.device)
            masks = masks.to(self.device)

            outputs = self.model(images)

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            # Dice
            intersection = (preds * masks).sum((1,2,3))
            union = preds.sum((1,2,3)) + masks.sum((1,2,3))
            dice = (2*intersection + 1e-6)/(union + 1e-6)

            # IoU
            union_iou = preds.sum((1,2,3)) + masks.sum((1,2,3)) - intersection
            iou = (intersection + 1e-6)/(union_iou + 1e-6)

            total_dice += dice.mean().item()
            total_iou += iou.mean().item()

            # Pixel Accuracy
            total_correct += (preds == masks).sum().item()
            total_pixels += masks.numel()

        avg_dice = total_dice / len(loader)
        avg_iou = total_iou / len(loader)
        accuracy = total_correct / total_pixels

        return avg_dice, avg_iou, accuracy

# MODULE 3.2: SEGMENTATION ARCHITECTURE & PIPELINE TRAINING (UNet++)

In [36]:
# =====================================================================
# MODULE 3: SEGMENTATION ARCHITECTURE & PIPELINE TRAINING (UNet++)
# =====================================================================

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp


# ---------------------------------------------------------------------
# Segmentation Model
# ---------------------------------------------------------------------

class UNetPlusPlus(nn.Module):
    """
    U-Net++ with ImageNet pretrained encoder.
    """

    def __init__(self):
        super().__init__()

        self.network = smp.UnetPlusPlus(
            encoder_name="resnet34",          # resnet50 also works
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
            activation=None
        )

    def forward(self, x):
        return self.network(x)


# ---------------------------------------------------------------------
# Dice Loss
# ---------------------------------------------------------------------

class DiceLoss(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, pred, target):

        pred = torch.sigmoid(pred)

        inter = (pred * target).sum((2,3))
        union = pred.sum((2,3)) + target.sum((2,3))

        dice = (2*inter + 1e-6)/(union + 1e-6)

        return 1 - dice.mean()


# ---------------------------------------------------------------------
# Training Engine
# ---------------------------------------------------------------------

class ExecutionEngine:

    def __init__(self, model, device):

        self.device = device
        self.model = model.to(device)

        self.bce_loss = nn.BCEWithLogitsLoss()
        self.dice_loss = DiceLoss()

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=1e-4,
            weight_decay=1e-4
        )


    def fit(self, loader):

        self.model.train()

        epoch_loss = 0

        for images, masks in loader:

            images = images.to(self.device)
            masks = masks.to(self.device)

            self.optimizer.zero_grad()

            outputs = self.model(images)

            loss = (
                self.bce_loss(outputs, masks)
                +
                self.dice_loss(outputs, masks)
            )

            loss.backward()

            self.optimizer.step()

            epoch_loss += loss.item()

        return epoch_loss / len(loader)


    @torch.no_grad()
    def evaluate(self, loader):

        self.model.eval()

        total_dice = 0
        total_iou = 0
        total_correct = 0
        total_pixels = 0

        for images, masks in loader:

            images = images.to(self.device)
            masks = masks.to(self.device)

            outputs = self.model(images)

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            intersection = (preds * masks).sum((1,2,3))

            pred_area = preds.sum((1,2,3))
            gt_area = masks.sum((1,2,3))

            dice = (2*intersection + 1e-6) / (
                pred_area + gt_area + 1e-6
            )

            union = pred_area + gt_area - intersection

            iou = (intersection + 1e-6) / (
                union + 1e-6
            )

            total_dice += dice.sum().item()
            total_iou += iou.sum().item()

            total_correct += (preds == masks).sum().item()
            total_pixels += masks.numel()

        avg_dice = total_dice / len(loader.dataset)
        avg_iou = total_iou / len(loader.dataset)
        accuracy = total_correct / total_pixels

        return avg_dice, avg_iou, accuracy

# MAIN EXPERIMENT EXECUTION PIPELINE

In [37]:
# # =====================================================================
# # MAIN EXPERIMENT EXECUTION PIPELINE
# # =====================================================================

# if __name__ == "__main__":
#     # 0. System Initialization
#     torch.manual_seed(42)
#     device = config.DEVICE
#     print(f"[System] Executing computational runtime on: {device}")

#     # 1. Dataset Parsing Configuration
#     # ADJUST THIS PATH TO MATCH YOUR UPLOADED ZIP FILE NAME
#     # ZIP_ARCHIVE_PATH = "Kvasir-SEG.zip"
#     ZIP_ARCHIVE_PATH = config.DATASET_ZIP

#     if not os.path.exists(ZIP_ARCHIVE_PATH):
#         raise FileNotFoundError(f"Please upload your Kaggle zip file to Colab and update ZIP_ARCHIVE_PATH to point to it.")

#     img_paths, mask_paths = prepare_kvasir_data(ZIP_ARCHIVE_PATH)
#     base_dataset = KvasirSegDataset(img_paths, mask_paths)

#     # 2. Strict Train/Test Separation Strategy (80/20)
#     num_samples = len(base_dataset)
#     indices = np.random.permutation(num_samples)
#     split_boundary = int(num_samples * 0.8)

#     train_indices = list(indices[:split_boundary])
#     test_indices = list(indices[split_boundary:])

#     test_loader = DataLoader(Subset(base_dataset, test_indices), batch_size=16, shuffle=False)

#     # 3. Generate Graph-Pruned Dataset Partition
#     pruner = DinoEmbeddingGraphPruner(base_dataset, train_indices, device)
#     # tau: Similarity filter cutoff. p: top-centrality retention budget per community.
#     pruned_train_indices = pruner.prune(tau=0.75, p=0.30)

#     # 4. Benchmarking Environment Execution
#     # Loader A: Full Baseline Training System
#     full_train_loader = DataLoader(Subset(base_dataset, train_indices), batch_size=16, shuffle=True)
#     # Loader B: Pruned Optimized System
#     pruned_train_loader = DataLoader(Subset(base_dataset, pruned_train_indices), batch_size=16, shuffle=True)

#     EPOCHS = 10  # Scale this as needed for your final thesis experiments

#     # --- EXPERIMENT 1: BASELINE WORKFLOW ---
#     print("\n[Pipeline] Training Model on FULL Kvasir Dataset...")
#     full_model = CompactUNet()
#     engine_full = ExecutionEngine(full_model, device)

#     t0 = time.time()
#     for epoch in range(EPOCHS):
#         engine_full.fit(full_train_loader)
#     t_full = time.time() - t0
#     dice_full, iou_full = engine_full.evaluate(test_loader)

#     # --- EXPERIMENT 2: PRUNED WORKFLOW ---
#     print("\n[Pipeline] Training Model on PRUNED Kvasir Dataset...")
#     pruned_model = CompactUNet()
#     engine_pruned = ExecutionEngine(pruned_model, device)

#     t1 = time.time()
#     for epoch in range(EPOCHS):
#         engine_pruned.fit(pruned_train_loader)
#     t_pruned = time.time() - t1
#     dice_pruned, iou_pruned = engine_pruned.evaluate(test_loader)

# FINAL STATISTICAL THESIS REPORTING

In [38]:
    # # =====================================================================
    # # FINAL STATISTICAL THESIS REPORTING
    # # =====================================================================
    # print("\n" + "="*50)
    # print("               FINAL EXPERIMENTAL REPORT")
    # print("="*50)
    # print(f"Original Training Set Count : {len(train_indices)}")
    # print(f"Pruned Training Set Count   : {len(pruned_train_indices)}")
    # print(f"Dataset Size Reduction Ratio: {((1.0 - len(pruned_train_indices)/len(train_indices))*100):.2f}%")
    # print("-"*50)
    # print(f"Full Dataset Train Time     : {t_full:.2f} seconds")
    # print(f"Pruned Dataset Train Time   : {t_pruned:.2f} seconds")
    # print(f"Training Acceleration delta : {(((t_full - t_pruned) / t_full) * 100):.2f}% Faster")
    # print("-"*50)
    # print(f"Full Dataset Metric Score   : Mean Dice: {dice_full:.4f} | Mean IoU: {iou_full:.4f}")
    # print(f"Pruned Dataset Metric Score : Mean Dice: {dice_pruned:.4f} | Mean IoU: {iou_pruned:.4f}")
    # print(f"Absolute Dice Variance Delta: {(dice_full - dice_pruned):.4f}")
    # print("="*50)

# Modify Approach - v1

In [39]:
# import torch
# import torch.nn as nn
# import numpy as np
# import networkx as nx
# from community import community_louvain
# from torch.utils.data import Dataset, DataLoader, Subset
# from torchvision import transforms

class AccuracyOptimizedGraphPruner:
    """Enhanced training-free dataset pruner using robust feature fusion."""
    def __init__(self, dataset: Dataset, train_indices: list, device: torch.device):
        self.dataset = dataset
        self.train_indices = train_indices
        self.device = device

        print("[DINOv2 Setup] Loading dinov2_vits14 for maximum feature extraction accuracy...")
        self.model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
        self.model.eval()

        # ImageNet standardization required for reliable ViT embeddings
        self.dino_transform = transforms.Compose([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    @torch.no_grad()
    def prune(self, tau: float = 0.75, p: float = 0.30) -> list:
        fused_embeddings = []
        print("[Feature Extraction] Building high-fidelity hybrid latent embeddings...")

        for idx in self.train_indices:
            img, _ = self.dataset[idx]
            input_tensor = self.dino_transform(img).unsqueeze(0).to(self.device)

            # Extract features from the intermediate layer blocks
            # DINOv2 forward features returns a dict with 'x_norm_clstoken' and 'x_norm_patchtokens'
            outputs = self.model.forward_features(input_tensor)

            cls_token = outputs['x_norm_clstoken'] # Shape: [1, 384]
            patch_tokens = outputs['x_norm_patchtokens'] # Shape: [1, 256, 384] for 224x224 input

            # Global Average Pooling over the spatial patch grid to pull out local textures
            gap_patches = torch.mean(patch_tokens, dim=1) # Shape: [1, 384]

            # Concat global context and localized features to capture polyp boundaries accurately
            hybrid_features = torch.cat((cls_token, gap_patches), dim=1) # Shape: [1, 768]
            hybrid_features = nn.functional.normalize(hybrid_features, p=2, dim=1)

            fused_embeddings.append(hybrid_features.cpu().numpy().flatten())

        fused_embeddings = np.array(fused_embeddings) # Matrix Shape: [N_train, 768]

        print("[Graph Theory] Generating topological similarity manifold...")
        # Compute exact cosine similarity via dot product matrix multiplication
        cosine_sim = np.dot(fused_embeddings, fused_embeddings.T)

        # Scale to standard safe bounds [0, 1]
        adj_matrix = (cosine_sim + 1.0) / 2.0

        # Binary adjacency conversion via indicator thresholding
        binary_edges = (adj_matrix >= tau).astype(int)

        # Topology assignment
        G = nx.from_numpy_array(binary_edges)

        print("[Louvain Detection] Partitions clustering to preserve minority edge cases...")
        partition = community_louvain.best_partition(G)

        communities = {}
        for node, comm_id in partition.items():
            communities.setdefault(comm_id, []).append(node)

        selected_indices = []
        for comm_id, nodes in communities.items():
            if len(nodes) <= 1:
                selected_indices.append(nodes[0])
            else:
                # Degree Centrality sorts robust, well-represented nodes first
                degrees = dict(G.degree(nodes))
                sorted_nodes = sorted(nodes, key=lambda n: degrees[n], reverse=True)

                # Dynamic budget allocation based on target retention parameter p
                budget = max(1, int(np.ceil(p * len(nodes))))
                selected_indices.extend(sorted_nodes[:budget])

        pruned_global_indices = [self.train_indices[i] for i in selected_indices]
        print(f"[Success] Retention complete. Selected {len(pruned_global_indices)} high-value structural samples.")
        return pruned_global_indices

## New Main

In [40]:
# =====================================================================
# MAIN EXPERIMENT EXECUTION PIPELINE
# =====================================================================

if __name__ == "__main__":

    # import os
    # import time
    # import torch
    # import numpy as np
    # from torch.utils.data import DataLoader, Subset

    # -------------------------------------------------------------
    # 0. System Initialization
    # -------------------------------------------------------------
    torch.manual_seed(42)
    np.random.seed(42)

    device = config.DEVICE
    print(f"[System] Running on: {device}")

    # -------------------------------------------------------------
    # 1. Dataset Preparation
    # -------------------------------------------------------------
    ZIP_ARCHIVE_PATH = config.DATASET_ZIP

    if not os.path.exists(ZIP_ARCHIVE_PATH):
        raise FileNotFoundError(
            f"Dataset not found at {ZIP_ARCHIVE_PATH}"
        )

    img_paths, mask_paths = prepare_kvasir_data(ZIP_ARCHIVE_PATH)
    base_dataset = KvasirSegDataset(img_paths, mask_paths)

    print(f"Total Images: {len(base_dataset)}")

    # -------------------------------------------------------------
    # 2. Train/Test Split (80/20)
    # -------------------------------------------------------------
    indices = np.random.permutation(len(base_dataset))

    split = int(0.8 * len(base_dataset))

    train_indices = list(indices[:split])
    test_indices = list(indices[split:])

    # train_indices = train_indices[:10]

    print(f"Training Images : {len(train_indices)}")
    print(f"Testing Images  : {len(test_indices)}")

    # -------------------------------------------------------------
    # 3. Test Loader
    # -------------------------------------------------------------
    test_loader = DataLoader(
        Subset(base_dataset, test_indices),
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # -------------------------------------------------------------
    # 4. Graph-Based Dataset Pruning
    # -------------------------------------------------------------
    print("\n==============================")
    print("Running Graph Dataset Pruning")
    print("==============================")

    pruner = AccuracyOptimizedGraphPruner(
        dataset=base_dataset,
        train_indices=train_indices,
        device=device
    )

    pruned_train_indices = pruner.prune(
        tau=config.SIMILARITY_THRESHOLD,
        p=config.RETENTION_RATIO
    )

    print(f"\nOriginal Training Samples : {len(train_indices)}")
    print(f"Pruned Training Samples   : {len(pruned_train_indices)}")
    print(f"Retention Ratio           : {100*len(pruned_train_indices)/len(train_indices):.2f}%")

    # -------------------------------------------------------------
    # 5. DataLoaders
    # -------------------------------------------------------------
    full_train_loader = DataLoader(
        Subset(base_dataset, train_indices),
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    pruned_train_loader = DataLoader(
        Subset(base_dataset, pruned_train_indices),
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    # -------------------------------------------------------------
    # 6. Training Configuration
    # -------------------------------------------------------------
    EPOCHS = config.EPOCHS

    # =============================================================
    # EXPERIMENT 1 : FULL DATASET
    # =============================================================
    print("\n===================================")
    print("Training on FULL Dataset")
    print("===================================")

    # full_model = CompactUNet()
    full_model = UNetPlusPlus()
    engine_full = ExecutionEngine(full_model, device)

    start = time.time()

    for epoch in range(EPOCHS):
        loss = engine_full.fit(full_train_loader)
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {loss:.4f}")

    full_training_time = time.time() - start

    dice_full, iou_full, acc_full = engine_full.evaluate(test_loader)

    # =============================================================
    # EXPERIMENT 2 : PRUNED DATASET
    # =============================================================
    print("\n===================================")
    print("Training on PRUNED Dataset")
    print("===================================")

    # pruned_model = CompactUNet()
    pruned_model = UNetPlusPlus()
    engine_pruned = ExecutionEngine(pruned_model, device)

    start = time.time()

    for epoch in range(EPOCHS):
        loss = engine_pruned.fit(pruned_train_loader)
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {loss:.4f}")

    pruned_training_time = time.time() - start

    dice_pruned, iou_pruned, acc_pruned = engine_pruned.evaluate(test_loader)

    # -------------------------------------------------------------
    # 7. Results
    # -------------------------------------------------------------
    print("\n===================================")
    print("FINAL RESULTS")
    print("===================================")

    # Final Parameters
    print(f"Similarity Threshold : {config.SIMILARITY_THRESHOLD}")
    print(f"Retention Ratio      : {config.RETENTION_RATIO}")

    print(f"Original Training Samples : {len(train_indices)}")
    print(f"Pruned Training Samples   : {len(pruned_train_indices)}")

    print("\nFull Dataset")
    print(f"Training Time : {full_training_time:.2f} sec")
    print(f"Dice Score    : {dice_full:.4f}")
    print(f"IoU Score     : {iou_full:.4f}")

    print("\nPruned Dataset")
    print(f"Training Time : {pruned_training_time:.2f} sec")
    print(f"Dice Score    : {dice_pruned:.4f}")
    print(f"IoU Score     : {iou_pruned:.4f}")

    print("\n===================================")
    print("FINAL REPORT")
    print("===================================")
    print("\nFull Dataset")
    print(f"Training Time : {full_training_time:.2f} sec")
    print(f"Accuracy      : {acc_full:.4f}")
    print(f"Dice Score    : {dice_full:.4f}")
    print(f"IoU Score     : {iou_full:.4f}")

    print("\nPruned Dataset")
    print(f"Training Time : {pruned_training_time:.2f} sec")
    print(f"Accuracy      : {acc_pruned:.4f}")
    print(f"Dice Score    : {dice_pruned:.4f}")
    print(f"IoU Score     : {iou_pruned:.4f}")


    print("\nSpeedup")
    print(f"{full_training_time/pruned_training_time:.2f}x")

    # total time
    print("\n===================================")
    print("TOTAL TIME")
    print("===================================")
    print(f"Full Dataset Training Time : {full_training_time:.2f} sec")
    print(f"Pruned Dataset Training Time : {pruned_training_time:.2f} sec")
    print(f"Total Time : {full_training_time + pruned_training_time:.2f} sec")
    print(f"Speedup : {full_training_time/pruned_training_time:.2f}x")
    print("\n===================================")


    print("\n===================================")
    print("END OF REPORT")
    print("===================================")

[System] Running on: cuda
[Data] Found 1000 images and 1000 segmentation masks.
Total Images: 1000
Training Images : 800
Testing Images  : 200

Running Graph Dataset Pruning
[DINOv2 Setup] Loading dinov2_vits14 for maximum feature extraction accuracy...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


[Feature Extraction] Building high-fidelity hybrid latent embeddings...
[Graph Theory] Generating topological similarity manifold...
[Louvain Detection] Partitions clustering to preserve minority edge cases...
[Success] Retention complete. Selected 168 high-value structural samples.

Original Training Samples : 800
Pruned Training Samples   : 168
Retention Ratio           : 21.00%

Training on FULL Dataset
Epoch [1/50] Loss: 0.7674
Epoch [2/50] Loss: 0.4457
Epoch [3/50] Loss: 0.3188
Epoch [4/50] Loss: 0.2510
Epoch [5/50] Loss: 0.1989
Epoch [6/50] Loss: 0.1675
Epoch [7/50] Loss: 0.1408
Epoch [8/50] Loss: 0.1318
Epoch [9/50] Loss: 0.1378
Epoch [10/50] Loss: 0.1222
Epoch [11/50] Loss: 0.1147
Epoch [12/50] Loss: 0.0938
Epoch [13/50] Loss: 0.0786
Epoch [14/50] Loss: 0.0695
Epoch [15/50] Loss: 0.0665
Epoch [16/50] Loss: 0.0602
Epoch [17/50] Loss: 0.0548
Epoch [18/50] Loss: 0.0514
Epoch [19/50] Loss: 0.0487
Epoch [20/50] Loss: 0.0469
Epoch [21/50] Loss: 0.0434
Epoch [22/50] Loss: 0.0406
Epoch

# End